In [1]:
import os

In [2]:
os.environ['HF_HOME'] = 'D:\\huggingface_cache'

In [3]:
%pwd

'd:\\Text-Summarizer-Project\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\Text-Summarizer-Project'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int



In [7]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

[2025-07-26 14:00:58,030: INFO: __init__: Logger is successfully configured!]


In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [9]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

c:\Users\ASUS\.conda\envs\textS\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-07-26 14:01:04,380: INFO: config: PyTorch version 2.4.1 available.]


In [10]:
# --- Fixes for IndexError: index out of range in self ---
# 1. Always re-tokenize the dataset with the current tokenizer before training.
# 2. Fix the indentation of preprocess_function and the re-tokenization block.
# 3. Add a check to ensure all splits exist and print a sample for debugging.

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        print("Inside train() method...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_t5 = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_t5)

        # loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # --- Always re-tokenize with the current tokenizer ---
        def preprocess_function(examples):
            inputs = examples["dialogue"] if "dialogue" in examples else examples["text"]
            model_inputs = tokenizer(inputs, max_length=512, truncation=True)
            with tokenizer.as_target_tokenizer():
                labels = tokenizer(examples["summary"], max_length=128, truncation=True)
            model_inputs["labels"] = labels["input_ids"]
            return model_inputs

        # Re-tokenize all splits to avoid index errors
        for split in ["train", "validation"]:
            if split in dataset_samsum_pt:
                dataset_samsum_pt[split] = dataset_samsum_pt[split].map(
                    preprocess_function,
                    batched=True,
                    remove_columns=dataset_samsum_pt[split].column_names,
                )

        # Optional: Print a sample to debug
        print("Sample input_ids:", dataset_samsum_pt["train"][0]["input_ids"][:10])
        print("Tokenizer vocab size:", tokenizer.vocab_size)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size, per_device_eval_batch_size=self.config.per_device_train_batch_size,
            weight_decay=self.config.weight_decay, logging_steps=self.config.logging_steps,
            evaluation_strategy=self.config.evaluation_strategy, eval_steps=(self.config.eval_steps), save_steps=1e6,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model=model_t5, args=trainer_args,
            tokenizer=tokenizer, data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"], 
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        # Save model and tokenizer
        model_t5.save_pretrained(os.path.join(self.config.root_dir, "t5-small-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))

In [11]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2025-07-26 14:01:05,286: INFO: common: yaml file: config\config.yaml loaded successfully]


[2025-07-26 14:01:05,291: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-26 14:01:05,293: INFO: common: created directory at: D:/Text-Summarizer-Project/artifacts]
[2025-07-26 14:01:05,294: INFO: common: created directory at: D:/Text-Summarizer-Project/artifacts/model_trainer]
[2025-07-26 14:01:05,293: INFO: common: created directory at: D:/Text-Summarizer-Project/artifacts]
[2025-07-26 14:01:05,294: INFO: common: created directory at: D:/Text-Summarizer-Project/artifacts/model_trainer]
Inside train() method...
Inside train() method...


Map:   0%|          | 0/818 [00:00<?, ? examples/s]c:\Users\ASUS\.conda\envs\textS\lib\site-packages\transformers\tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 818/818 [00:00<00:00, 6560.43 examples/s]c:\Users\ASUS\.conda\envs\textS\lib\site-packages\transformers\tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Map: 100%|██████████| 818/818 [00:00<00:00, 5825.30 examples/s]
c:

Sample input_ids: [21542, 10, 27, 13635, 5081, 5, 531, 25, 241, 128]
Tokenizer vocab size: 32100


  0%|          | 0/920 [00:00<?, ?it/s]Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  1%|          | 10/920 [00:55<1:19:34,  5.25s/it]

...

{'loss': 1.945, 'grad_norm': 47.189849853515625, 'learning_rate': 1.1904761904761904e-06, 'epoch': 0.99}


100%|██████████| 920/920 [1:23:46<00:00,  5.32s/it]

{'loss': 2.0139, 'grad_norm': 44.68335723876953, 'learning_rate': 0.0, 'epoch': 1.0}


100%|██████████| 920/920 [1:23:47<00:00,  5.47s/it]



{'train_runtime': 5027.855, 'train_samples_per_second': 2.93, 'train_steps_per_second': 0.183, 'train_loss': 2.2544831545456594, 'epoch': 1.0}
